# <font color='#BFD72F' size=6>**7. Model stability over time**</font> <a class="anchor" id="stability"></a>

[Back to TOC](#toc)

**Why a separate analysis?** The cumulative scores in section 6 average 70k hours into one number, so they hide
*when* a model is good and when it is not. In a stream, the question is whether a model keeps the same quality as
the data change, and whether it adapts when they do.

**Protocol.** Every model is replayed again with exactly the same prequential protocol as section 6
(shared warm-up, then predict -> evaluate -> learn, tuned hyperparameters). This time **every single prediction is
logged**, so any stratification (window, year, season, event onset) is computed from the same predictions and
not from re-runs. The stochastic ensembles are also replayed with several seeds.

| Dimension | Question | Measure |
|---|---|---|
| Temporal variability | How much does quality move from window to window? | rolling macro-F1 over informative windows: IQR, std, worst 5 % (p05) |
| Long-term trend | Does the model degrade or improve over the years? | macro-F1 per year, OLS slope per year + Kendall τ |
| Seasonality | Is quality season-dependent? | metrics per season; false alarms in event-free windows |
| Relative value over time | Does it beat persistence, and when? | windowed κ-t; share of windows beating No-change |
| Event-level behaviour | Does it catch the *start* of an event, and is this stable? | onset hit rate, detection delay, missed events, spurious alarm episodes, per year |
| Drift and adaptation | When does the error change and does the model recover? | ADWIN on each model's 0/1 error stream; ARF tree replacements; recovery time |
| Data drift | Did the inputs or the event rate shift? | KS statistic and standardised mean shift per feature and year (hot season); event rate per year |
| Randomness | Is the conclusion robust to the seed? | spread of the ensembles across seeds |
| Internal stability | Do the learned weights settle? | logistic-regression coefficient settling time and sign changes |
| Sensitivity | Does the conclusion depend on the window length? | variability for 7, 30 and 90-day windows |

**Informative window.** A rolling window in which fewer than 5 % of the hours are events (all winter) has an undefined
or degenerate event F1. Those windows are excluded from the rolling quality metrics, and false alarms are reported
separately for them, so the curves show model behaviour and not the season.

In [ ]:
# ---- CELL S0: configuration (needs cells 0-2, the evaluator cell, P0 and T0-T3) ----
import math
import pickle
import hashlib

from IPython.display import Markdown, display
from joblib import Parallel, delayed
from scipy import stats as sps
from river import drift, forest

STAB_USE_TUNED = True                      # same configuration as the final evaluation (T3)
STAB_SEEDS = [RANDOM_SEED, 7, 123, 2024, 31415]   # first one must be RANDOM_SEED (= the main run)
STAB_STOCHASTIC = ["Online Bagging", "Adaptive Random Forest"]   # the only models whose output depends on the seed
STAB_WINDOW = WINDOW_SIZE                  # 720 h = 30 days, same as section 6
STAB_WINDOW_GRID = [24 * 7, WINDOW_SIZE, 24 * 90]   # sensitivity of the conclusions to the window length
STAB_STRIDE = 24                           # one rolling value per day
STAB_MIN_EVENT_SHARE = 0.05                # informative window: at least 5 % event hours
STAB_MIN_TRANSITIONS = 4                   # windowed kappa-t needs label changes, otherwise persistence is trivially perfect
STAB_MIN_YEAR_SHARE = 0.9                  # a year enters the trend test only if at least 90 % of its hours are scored
ADWIN_DELTA = 0.002                        # River default; smaller = fewer, more confident detections
RECOVERY_WINDOW = 24 * 7                   # error measured over 7 days when checking recovery after a drift
RECOVERY_MAX_DAYS = 90
HOT_SEASON_MONTHS = [5, 6, 7, 8, 9, 10]    # where the event happens (see the monthly profile in P7)
TRANSITION_MONTHS = [4, 5, 6, 9, 10, 11]   # season changes: where seasonal drift is expected
SEASONS = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
           6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]
STAB_N_JOBS = 1                            # -1 = all cores (the replay is independent per city/representation/seed)
STAB_FORCE_RERUN = False                   # True ignores the cache
NC = "No-change (baseline)"
MAJ = "Majority class (baseline)"

STAB_DIR = RESULTS_DIR / "stability"
STAB_DIR.mkdir(parents=True, exist_ok=True)
STAB_CACHE = STAB_DIR / "stability_replay_cache.pkl"

_required = ["splits", "REPRESENTATIONS", "CITIES", "TARGET_COL", "MODEL_SPECS", "make_models", "iter_events",
             "RESULTS_DIR", "FIGURES_DIR", "show_and_save", "model_style", "MODEL_COLORS"]
_missing = [n for n in _required if n not in globals()]
assert not _missing, f"Run the earlier cells first, missing: {_missing}"
assert not STAB_USE_TUNED or "params_for" in globals(), "Run T2 first (params_for), or set STAB_USE_TUNED = False"
assert STAB_SEEDS[0] == RANDOM_SEED, "The first seed must be RANDOM_SEED so the main run matches section 6"
assert all(m in MODEL_SPECS for m in STAB_STOCHASTIC)

print(f"Rolling window {STAB_WINDOW} h, stride {STAB_STRIDE} h, informative if >= {STAB_MIN_EVENT_SHARE:.0%} event hours")
print(f"Seeds for {STAB_STOCHASTIC}: {STAB_SEEDS}")

In [ ]:
# ---- CELL S1: replay with per-event logging, error-drift detection and seed repetitions ----
def _stab_params(city: str, representation: str) -> dict:
    return params_for(city, representation) if STAB_USE_TUNED else {}


def _build_seeded(name: str, params: dict, seed: int):
    spec = MODEL_SPECS[name]
    return spec["build"](**{**spec["default"], **params.get(name, {}), "seed": seed})


def replay_logged(city: str, representation: str, seed: int | None = None, track_drift: bool = True) -> dict:
    """Same protocol as track_online_performance (warm-up learn -> predict -> evaluate -> learn),
    but every prediction is stored. seed=None: all models of section 6; otherwise only the stochastic
    ensembles, rebuilt with that seed."""
    split, cols = splits[city], REPRESENTATIONS[representation]
    params = _stab_params(city, representation)
    if seed is None:
        models = make_models(params)
    else:
        models = {name: _build_seeded(name, params, seed) for name in STAB_STOCHASTIC}

    for _, x, y in iter_events(split["warmup"], cols):
        for model in models.values():
            model.learn_one(x, y)

    n = len(split["evaluation"])
    preds = {name: np.full(n, -1, dtype=np.int8) for name in models}          # -1 = model could not predict
    detectors = {name: drift.ADWIN(delta=ADWIN_DELTA) for name in models} if track_drift else {}
    error_drifts = {name: [] for name in models}                              # (index, error before, error after)
    arf_names = [name for name, m in models.items() if isinstance(m, forest.ARFClassifier)]
    arf_seen = {name: models[name].n_drifts_detected() for name in arf_names}
    arf_swaps = {name: [] for name in arf_names}                              # indices where ARF replaced a tree

    for i, (_, x, y) in enumerate(iter_events(split["evaluation"], cols)):
        for name, model in models.items():
            y_pred = model.predict_one(x)                                     # 1. predict
            if y_pred is not None:
                preds[name][i] = int(y_pred)
            if track_drift:                                                   # 2. evaluate (error stream)
                det = detectors[name]
                before = det.estimation
                det.update(1 if y_pred is None else int(int(y_pred) != y))
                if det.drift_detected:
                    error_drifts[name].append((i, float(before), float(det.estimation)))
            model.learn_one(x, y)                                             # 3. learn
            if name in arf_seen:
                k = model.n_drifts_detected()
                if k > arf_seen[name]:
                    arf_swaps[name].extend([i] * (k - arf_seen[name]))
                    arf_seen[name] = k

    return dict(city=city, representation=representation, seed=seed, preds=preds,
                error_drifts=error_drifts, arf_swaps=arf_swaps)


def _cache_key() -> str:
    payload = dict(
        cities=CITIES, reps={r: list(c) for r, c in REPRESENTATIONS.items()}, seeds=STAB_SEEDS,
        stochastic=STAB_STOCHASTIC, adwin=ADWIN_DELTA, tuned=STAB_USE_TUNED,
        params={f"{c}|{r}": _stab_params(c, r) for c in CITIES for r in REPRESENTATIONS},
        data={c: [len(splits[c]["warmup"]), len(splits[c]["evaluation"]),
                  str(splits[c]["evaluation"]["time"].iloc[0]), str(splits[c]["evaluation"]["time"].iloc[-1]),
                  int(splits[c]["evaluation"][TARGET_COL].sum())] for c in CITIES},
    )
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()


def run_stability_replays() -> dict:
    key = _cache_key()
    if STAB_CACHE.exists() and not STAB_FORCE_RERUN:
        with open(STAB_CACHE, "rb") as f:
            cached = pickle.load(f)
        if cached.get("key") == key:
            print(f"Loaded replay from cache ({STAB_CACHE.name})")
            return cached["runs"]
    tasks = [(c, r, None, True) for c in CITIES for r in REPRESENTATIONS]
    tasks += [(c, r, s, False) for c in CITIES for r in REPRESENTATIONS for s in STAB_SEEDS[1:]]
    t0 = time.perf_counter()
    out = Parallel(n_jobs=STAB_N_JOBS, verbose=5)(delayed(replay_logged)(*t) for t in tasks)
    print(f"{len(tasks)} replays in {time.perf_counter() - t0:,.0f} s")
    with open(STAB_CACHE, "wb") as f:
        pickle.dump({"key": key, "runs": out}, f)
    return out


def assemble_runs(raw_runs: list[dict]) -> dict:
    """stab[(city, rep)] = time, y, y_prev0, preds{model}, seed_preds{model}{seed}, error_drifts, arf_swaps."""
    stab = {}
    for city in CITIES:
        ev = splits[city]["evaluation"]
        y_prev0 = int(splits[city]["warmup"][TARGET_COL].iloc[-1]) if len(splits[city]["warmup"]) else int(ev[TARGET_COL].iloc[0])
        for rep in REPRESENTATIONS:
            stab[(city, rep)] = dict(time=ev["time"].to_numpy(), y=ev[TARGET_COL].to_numpy(dtype=np.int8),
                                     y_prev0=y_prev0, preds={}, seed_preds={m: {} for m in STAB_STOCHASTIC},
                                     error_drifts={}, arf_swaps={})
    for run in raw_runs:
        entry = stab[(run["city"], run["representation"])]
        if run["seed"] is None:
            entry["preds"] = run["preds"]
            entry["error_drifts"] = run["error_drifts"]
            entry["arf_swaps"] = run["arf_swaps"]
            for m in STAB_STOCHASTIC:
                entry["seed_preds"][m][STAB_SEEDS[0]] = run["preds"][m]
        else:
            for m, p in run["preds"].items():
                entry["seed_preds"][m][run["seed"]] = p
    return stab


stab = assemble_runs(run_stability_replays())
STAB_MODELS = [m for m in MODEL_COLORS if m in stab[(CITIES[0], list(REPRESENTATIONS)[0])]["preds"]]
STAB_LEARNERS = [m for m in STAB_MODELS if "baseline" not in m]

# consistency check: the logged run must reproduce the cumulative scores of section 6 (T3, tuned)
if "comparison_tuned" in globals() and STAB_USE_TUNED:
    for (city, rep), e in stab.items():
        for m in STAB_MODELS:
            p, y = e["preds"][m], e["y"]
            ok = p >= 0
            tp = int(((p == 1) & (y == 1) & ok).sum())
            ref = comparison_tuned.query("city == @city and representation == @rep and model == @m")
            if len(ref):
                assert tp == int(ref["tp"].iloc[0]), f"{city} {rep} {m}: replay differs from T3 (tp {tp} vs {int(ref['tp'].iloc[0])})"
    print("Replay reproduces the T3 confusion matrices exactly.")

In [ ]:
# ---- CELL S2: metric engine (vectorised; works for windows, years, seasons) ----
def _safe_div(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return np.divide(a, b, out=np.full(np.broadcast(a, b).shape, np.nan), where=b > 0)


def scores_from_counts(tp, fp, fn, tn, nc_ok, transitions, min_transitions=1) -> dict:
    """Event-class and macro scores from confusion counts. Undefined -> NaN (never a fake 0 or 0.5).
    nc_ok = events where 'repeat the previous true label' is right, counted on the same scored events."""
    tp, fp, fn, tn, nc_ok, transitions = (np.asarray(v, dtype=float) for v in (tp, fp, fn, tn, nc_ok, transitions))
    n = tp + fp + fn + tn
    f1_event = _safe_div(2 * tp, 2 * tp + fp + fn)
    f1_none = _safe_div(2 * tn, 2 * tn + fp + fn)
    accuracy = _safe_div(tp + tn, n)
    acc_nc = _safe_div(nc_ok, n)
    kappa_t = np.where((transitions >= min_transitions) & (acc_nc < 1), _safe_div(accuracy - acc_nc, 1 - acc_nc), np.nan)
    return {
        "n": n, "events": tp + fn, "event_rate": _safe_div(tp + fn, n),
        "macro_f1": (f1_event + f1_none) / 2, "event_f1": f1_event,
        "recall": _safe_div(tp, tp + fn), "precision": _safe_div(tp, tp + fp),
        "false_alarm_rate": _safe_div(fp, fp + tn), "accuracy": accuracy, "nc_accuracy": acc_nc,
        "kappa_t": kappa_t, "transitions": transitions,
    }


def _indicators(y: np.ndarray, p: np.ndarray, y_prev0: int) -> dict:
    ok = p >= 0
    prev = np.r_[y_prev0, y[:-1]]
    return dict(tp=ok & (y == 1) & (p == 1), fp=ok & (y == 0) & (p == 1), fn=ok & (y == 1) & (p == 0),
                tn=ok & (y == 0) & (p == 0), nc_ok=ok & (prev == y), transitions=prev != y, true_events=y == 1)


def _rolling_sum(a: np.ndarray, window: int) -> np.ndarray:
    c = np.concatenate(([0], np.cumsum(a, dtype=np.int64)))
    end = np.arange(1, len(a) + 1)
    return c[end] - c[np.maximum(0, end - window)]


def rolling_scores(time_, y, p, y_prev0, window=STAB_WINDOW, stride=STAB_STRIDE) -> pd.DataFrame:
    ind = _indicators(y, p, y_prev0)
    idx = np.unique(np.r_[np.arange(window - 1, len(y), stride), len(y) - 1])     # full windows only
    sums = {k: _rolling_sum(v, window)[idx] for k, v in ind.items()}
    sc = scores_from_counts(sums["tp"], sums["fp"], sums["fn"], sums["tn"], sums["nc_ok"], sums["transitions"],
                            STAB_MIN_TRANSITIONS)
    out = pd.DataFrame({"time": time_[idx], **sc, "fp": sums["fp"], "true_events": sums["true_events"]})
    out["informative"] = out["true_events"] >= math.ceil(STAB_MIN_EVENT_SHARE * window)
    out["false_alarms_per_30d"] = out["fp"] / out["n"] * 720
    return out


def grouped_scores(time_, y, p, y_prev0, by: str) -> pd.DataFrame:
    """by = 'year' | 'season' | 'year_season'. Scores from the summed confusion counts of each group."""
    t = pd.DatetimeIndex(time_)
    keys = {"year": t.year, "season": t.month.map(SEASONS)}
    if by == "year_season":
        group = {"year": keys["year"], "season": keys["season"]}
    else:
        group = {by: keys[by]}
    ind = _indicators(y, p, y_prev0)
    df = pd.DataFrame({k: v.astype(int) for k, v in ind.items()})
    df["hours"] = 1
    for k, v in group.items():
        df[k] = np.asarray(v)
    g = df.groupby(list(group)).sum()
    sc = scores_from_counts(g["tp"], g["fp"], g["fn"], g["tn"], g["nc_ok"], g["transitions"], 1)
    return g[["hours", "fp"]].assign(**{k: np.asarray(v) for k, v in sc.items()}).reset_index()


# ---- rolling curves for every experiment and model (+ every seed of the ensembles) ----
rolling_rows, seed_rolling_rows = [], []
for (city, rep), e in stab.items():
    for m in STAB_MODELS:
        r = rolling_scores(e["time"], e["y"], e["preds"][m], e["y_prev0"])
        rolling_rows.append(r.assign(city=city, representation=rep, model=m))
    for m, by_seed in e["seed_preds"].items():
        for s, p in by_seed.items():
            r = rolling_scores(e["time"], e["y"], p, e["y_prev0"])
            seed_rolling_rows.append(r[["time", "macro_f1", "informative"]].assign(city=city, representation=rep, model=m, seed=s))
rolling = pd.concat(rolling_rows, ignore_index=True)
seed_rolling = pd.concat(seed_rolling_rows, ignore_index=True)

# relative to persistence, window by window
_nc = rolling[rolling["model"] == NC][["city", "representation", "time", "macro_f1", "kappa_t"]]
rolling = rolling.merge(_nc.rename(columns={"macro_f1": "nc_macro_f1", "kappa_t": "nc_kappa_t"}),
                        on=["city", "representation", "time"], how="left")
rolling["delta_vs_nc"] = rolling["macro_f1"] - rolling["nc_macro_f1"]

# ---- per year, per season, per year x season ----
def _grouped_all(by):
    rows = []
    for (city, rep), e in stab.items():
        for m in STAB_MODELS:
            rows.append(grouped_scores(e["time"], e["y"], e["preds"][m], e["y_prev0"], by)
                        .assign(city=city, representation=rep, model=m))
    return pd.concat(rows, ignore_index=True)


yearly, seasonal, year_season = _grouped_all("year"), _grouped_all("season"), _grouped_all("year_season")
for _t in (yearly, seasonal, year_season):     # a period with almost no events has a degenerate event F1
    _t["informative"] = _t["event_rate"] >= STAB_MIN_EVENT_SHARE
_max_hours = yearly.groupby(["city", "representation", "model"])["hours"].transform("max")
yearly["complete_year"] = yearly["hours"] >= STAB_MIN_YEAR_SHARE * _max_hours
yearly = yearly.merge(yearly[yearly["model"] == NC][["city", "representation", "year", "macro_f1"]]
                      .rename(columns={"macro_f1": "nc_macro_f1"}), on=["city", "representation", "year"])
yearly["delta_vs_nc"] = yearly["macro_f1"] - yearly["nc_macro_f1"]

print(f"Rolling: {len(rolling):,} rows | informative windows: "
      f"{rolling.drop_duplicates(['city', 'representation', 'time'])['informative'].mean():.0%}")
display(yearly.pivot_table(index=["city", "representation", "model"], columns="year", values="macro_f1").round(3))

In [ ]:
# ---- CELL S3: event-level stability (onsets, delays, missed events, spurious alarms) ----
def _runs(mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    m = np.r_[False, np.asarray(mask, dtype=bool), False]
    d = np.diff(m.astype(np.int8))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1) - 1          # inclusive start / end


def event_table(e: dict, model: str) -> pd.DataFrame:
    """One row per true event episode: was its first hour predicted, how late was the first alarm, was it missed?"""
    y, p, t = e["y"], e["preds"][model], e["time"]
    hit = p == 1
    rows = []
    for a, b in zip(*_runs(y == 1)):
        if a == 0 and e["y_prev0"] == 1:
            continue                                    # episode started in the warm-up: onset not observed
        seg = hit[a:b + 1]
        first = int(np.argmax(seg)) if seg.any() else None
        rows.append(dict(onset=t[a], year=pd.Timestamp(t[a]).year, duration_h=b - a + 1,
                         onset_hit=bool(seg[0]), delay_h=np.nan if first is None else first,
                         missed=first is None, coverage=float(seg.mean())))
    return pd.DataFrame(rows)


def spurious_alarm_table(e: dict, model: str) -> pd.DataFrame:
    """Alarm episodes (consecutive predicted-event hours) that never overlap a true event."""
    y, p, t = e["y"], e["preds"][model], e["time"]
    rows = []
    for a, b in zip(*_runs(p == 1)):
        rows.append(dict(start=t[a], year=pd.Timestamp(t[a]).year, duration_h=b - a + 1,
                         spurious=not bool(y[a:b + 1].any())))
    return pd.DataFrame(rows, columns=["start", "year", "duration_h", "spurious"])


ev_rows, alarm_rows = [], []
for (city, rep), e in stab.items():
    for m in STAB_MODELS:
        ev_rows.append(event_table(e, m).assign(city=city, representation=rep, model=m))
        alarm_rows.append(spurious_alarm_table(e, m).assign(city=city, representation=rep, model=m))
events = pd.concat(ev_rows, ignore_index=True)
alarms = pd.concat(alarm_rows, ignore_index=True)

keys = ["city", "representation", "model"]
event_summary = events.groupby(keys).agg(
    episodes=("onset", "size"), onset_hit_rate=("onset_hit", "mean"), missed_share=("missed", "mean"),
    median_delay_h=("delay_h", "median"), mean_coverage=("coverage", "mean")).reset_index()
_spur = alarms.groupby(keys).agg(alarm_episodes=("spurious", "size"), spurious_episodes=("spurious", "sum")).reset_index()
_years = yearly.groupby(keys)["year"].nunique().rename("n_years").reset_index()
event_summary = event_summary.merge(_spur, on=keys, how="left").merge(_years, on=keys)
event_summary[["alarm_episodes", "spurious_episodes"]] = event_summary[["alarm_episodes", "spurious_episodes"]].fillna(0)
event_summary["spurious_episodes_per_year"] = event_summary["spurious_episodes"] / event_summary["n_years"]

events_yearly = events.groupby(keys + ["year"]).agg(
    episodes=("onset", "size"), onset_hit_rate=("onset_hit", "mean"), missed_share=("missed", "mean"),
    median_delay_h=("delay_h", "median")).reset_index()

display(event_summary.round(3))

In [ ]:
# ---- CELL S4: drift — model error drift (ADWIN), recovery, ARF tree swaps, target and input drift ----
def _trailing_error(y, p, window):
    err = ((p != y) | (p < 0)).astype(float)
    return _rolling_sum(err, window) / np.minimum(np.arange(1, len(err) + 1), window)


drift_rows = []
for (city, rep), e in stab.items():
    month = pd.DatetimeIndex(e["time"]).month
    for m in STAB_MODELS:
        roll_err = _trailing_error(e["y"], e["preds"][m], RECOVERY_WINDOW)
        for i, before, after in e["error_drifts"].get(m, []):
            worse = after > before
            recovery = np.nan
            if worse:
                start = i + RECOVERY_WINDOW - 1                     # first window entirely after the detection
                stop = min(len(roll_err), i + RECOVERY_MAX_DAYS * 24)
                below = np.flatnonzero(roll_err[start:stop] <= before) if start < stop else []
                recovery = (start + below[0] - i) / 24 if len(below) else np.nan   # >= 7 days by construction
            drift_rows.append(dict(city=city, representation=rep, model=m, kind="error drift", time=e["time"][i],
                                   index=i, error_before=before, error_after=after,
                                   direction="worse" if worse else "better", recovery_days=recovery,
                                   season=SEASONS[month[i]], in_transition_month=month[i] in TRANSITION_MONTHS))
        for i in e["arf_swaps"].get(m, []):
            drift_rows.append(dict(city=city, representation=rep, model=m, kind="ARF tree replaced", time=e["time"][i],
                                   index=i, season=SEASONS[month[i]], in_transition_month=month[i] in TRANSITION_MONTHS))
drifts = pd.DataFrame(drift_rows)

# change points in the event rate itself (prior drift): ADWIN on the DAILY event rate
# (on the hourly 0/1 target ADWIN fires on every diurnal swing, which is not drift)
target_drift_rows = []
for city in CITIES:
    e = stab[(city, list(REPRESENTATIONS)[0])]
    daily = pd.Series(e["y"], index=pd.DatetimeIndex(e["time"])).resample("D").mean().dropna()
    det = drift.ADWIN(delta=ADWIN_DELTA)
    for i, (day, rate) in enumerate(daily.items()):
        before = det.estimation
        det.update(float(rate))
        if det.drift_detected:
            target_drift_rows.append(dict(city=city, time=day, index=i, rate_before=before,
                                          rate_after=det.estimation))
target_drifts = pd.DataFrame(target_drift_rows, columns=["city", "time", "index", "rate_before", "rate_after"])

drift_summary = pd.DataFrame(columns=keys)
if len(drifts):
    _err = drifts[drifts["kind"] == "error drift"]
    _worse = _err[_err["direction"] == "worse"]
    drift_summary = (_err.groupby(keys)
                     .agg(error_drifts=("index", "size"), share_in_transition_months=("in_transition_month", "mean"))
                     .join(_worse.groupby(keys).agg(worsening=("index", "size"),
                                                    median_recovery_days=("recovery_days", "median"),
                                                    not_recovered=("recovery_days", lambda s: int(s.isna().sum()))))
                     .reset_index())
    drift_summary[["worsening", "not_recovered"]] = drift_summary[["worsening", "not_recovered"]].fillna(0)
    _swaps = drifts[drifts["kind"] == "ARF tree replaced"].groupby(keys).size().rename("arf_tree_replacements")
    drift_summary = drift_summary.merge(_swaps.reset_index(), on=keys, how="left")


# covariate drift: each year's hot season vs the reference hot season (warm-up; first evaluation year as fallback)
def covariate_drift(city: str) -> pd.DataFrame:
    features = sorted(set().union(*map(set, REPRESENTATIONS.values())))
    ev, wu = splits[city]["evaluation"], splits[city]["warmup"]
    hot = lambda df: df[df["time"].dt.month.isin(HOT_SEASON_MONTHS)]
    ref, ref_name = hot(wu), "warm-up"
    if len(ref) < 24 * 30:
        first_year = ev["time"].dt.year.min()
        ref, ref_name = hot(ev[ev["time"].dt.year == first_year]), f"evaluation {first_year}"
    rows = []
    for year, g in hot(ev).groupby(ev["time"].dt.year):
        for f in features:
            sd = ref[f].std()
            rows.append(dict(city=city, year=year, feature=f, reference=ref_name,
                             ks_stat=sps.ks_2samp(ref[f], g[f]).statistic,
                             std_mean_shift=(g[f].mean() - ref[f].mean()) / sd if sd else np.nan))
    return pd.DataFrame(rows)


cov_drift = pd.concat([covariate_drift(c) for c in CITIES], ignore_index=True)
prior_drift = (yearly[yearly["model"] == NC][["city", "year", "hours", "event_rate"]]
               .drop_duplicates(["city", "year"]))
display(drift_summary.round(3))
display(prior_drift.pivot_table(index="city", columns="year", values="event_rate").round(3))

In [ ]:
# ---- CELL S5: randomness (seeds), window-length sensitivity, coefficient stability ----
# seeds: overall macro-F1 of every seed + how far apart the rolling curves are
seed_rows = []
for (city, rep), e in stab.items():
    for m, by_seed in e["seed_preds"].items():
        for s, p in by_seed.items():
            ind = _indicators(e["y"], p, e["y_prev0"])
            sc = scores_from_counts(*(ind[k].sum() for k in ("tp", "fp", "fn", "tn", "nc_ok", "transitions")))
            seed_rows.append(dict(city=city, representation=rep, model=m, seed=s,
                                  macro_f1=float(sc["macro_f1"]), event_f1=float(sc["event_f1"]),
                                  recall=float(sc["recall"]), kappa_t=float(sc["kappa_t"])))
seed_scores = pd.DataFrame(seed_rows)
_band = (seed_rolling[seed_rolling["informative"]].groupby(["city", "representation", "model", "time"])["macro_f1"]
         .agg(lambda s: s.max() - s.min()).groupby(level=[0, 1, 2]).mean().rename("mean_band_width"))
seed_summary = (seed_scores.groupby(["city", "representation", "model"])
                .agg(seeds=("seed", "size"), macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
                     macro_f1_min=("macro_f1", "min"), macro_f1_max=("macro_f1", "max"))
                .join(_band).reset_index())

# window-length sensitivity of the variability measures
sens_rows = []
for W in STAB_WINDOW_GRID:
    for (city, rep), e in stab.items():
        nc_r = rolling_scores(e["time"], e["y"], e["preds"][NC], e["y_prev0"], window=W)
        for m in STAB_MODELS:
            r = rolling_scores(e["time"], e["y"], e["preds"][m], e["y_prev0"], window=W)
            inf = r["informative"] & r["macro_f1"].notna()
            v = r.loc[inf, "macro_f1"]
            sens_rows.append(dict(window_days=W // 24, city=city, representation=rep, model=m,
                                  mean=v.mean(), iqr=v.quantile(.75) - v.quantile(.25), p05=v.quantile(.05),
                                  share_beats_nc=(r.loc[inf, "macro_f1"] > nc_r.loc[inf, "macro_f1"]).mean()))
window_sensitivity = pd.DataFrame(sens_rows)
# is the stability ranking the same for every window length?
_rk = window_sensitivity[window_sensitivity["model"].isin(STAB_LEARNERS)].copy()
_rk["iqr_rank"] = _rk.groupby(["window_days", "city", "representation"])["iqr"].rank()
rank_agreement = (_rk.pivot_table(index=["city", "representation", "model"], columns="window_days", values="iqr_rank")
                  .corr(method="kendall"))

# logistic-regression coefficients: do they settle, and do they change sign?
def coefficient_stability() -> pd.DataFrame:
    if "weight_histories" not in globals():
        if "logistic_weight_history" not in globals():
            print("Run F0/F1 for the coefficient analysis.")
            return pd.DataFrame()
        globals()["weight_histories"] = {(c, r): logistic_weight_history(c, r) for c in CITIES for r in REPRESENTATIONS}
    rows = []
    for (city, rep), h in weight_histories.items():
        t = pd.DatetimeIndex(pd.to_datetime(h["time"]))
        for col in [c for c in h.columns if c.startswith("w:")]:
            w = h[col].to_numpy()
            final = w[-1]
            tol = 0.10 * max(abs(final), 0.05)                       # ±10 % of the final value (min. band 0.005)
            outside = np.flatnonzero(np.abs(w - final) > tol)
            settle = (t[outside[-1]] - t[0]).days if len(outside) else 0
            second_half = w[len(w) // 2:]
            signs = np.sign(w[np.abs(w) > tol])
            rows.append(dict(city=city, representation=rep, feature=col[2:], final_coef=final,
                             settle_days=settle, settled_share_of_stream=1 - settle / max(1, (t[-1] - t[0]).days),
                             std_second_half=second_half.std(),
                             sign_changes=int((np.diff(signs) != 0).sum()) if len(signs) > 1 else 0))
    return pd.DataFrame(rows)


coef_stability = coefficient_stability()
display(seed_summary.round(4))
display(window_sensitivity.pivot_table(index=["city", "representation", "model"], columns="window_days",
                                       values="iqr").round(3))
print("Kendall agreement of the IQR stability ranking between window lengths:")
display(rank_agreement.round(2))
if len(coef_stability):
    display(coef_stability.round(3))

In [ ]:
# ---- CELL S6: stability summary table (one row per city x representation x model) ----
def trend_test(df: pd.DataFrame) -> pd.Series:
    d = df[df["complete_year"] & df["macro_f1"].notna()]
    if len(d) < 4:
        return pd.Series(dict(trend_slope_per_year=np.nan, trend_p=np.nan, kendall_tau=np.nan, kendall_p=np.nan))
    lr = sps.linregress(d["year"], d["macro_f1"])
    kt = sps.kendalltau(d["year"], d["macro_f1"])
    return pd.Series(dict(trend_slope_per_year=lr.slope, trend_p=lr.pvalue,
                          kendall_tau=kt.statistic, kendall_p=kt.pvalue))


def summarise_stability() -> pd.DataFrame:
    rows = []
    for (city, rep, m), r in rolling.groupby(keys):
        inf = r[r["informative"] & r["macro_f1"].notna()]
        off = r[r["true_events"] == 0]
        yr = yearly[(yearly["city"] == city) & (yearly["representation"] == rep) & (yearly["model"] == m)]
        se = seasonal[(seasonal["city"] == city) & (seasonal["representation"] == rep) & (seasonal["model"] == m)]
        se_inf = se[se["informative"]]
        e = stab[(city, rep)]
        ind = _indicators(e["y"], e["preds"][m], e["y_prev0"])
        overall = scores_from_counts(*(ind[k].sum() for k in ("tp", "fp", "fn", "tn", "nc_ok", "transitions")))
        complete = yr[yr["complete_year"]]
        rows.append(dict(
            city=city, representation=rep, model=m,
            macro_f1_overall=float(overall["macro_f1"]), kappa_t_overall=float(overall["kappa_t"]),
            rolling_mean=inf["macro_f1"].mean(), rolling_std=inf["macro_f1"].std(),
            rolling_iqr=inf["macro_f1"].quantile(.75) - inf["macro_f1"].quantile(.25),
            rolling_p05=inf["macro_f1"].quantile(.05), rolling_min=inf["macro_f1"].min(),
            rolling_cv=inf["macro_f1"].std() / inf["macro_f1"].mean(),
            share_windows_beat_nc=(inf["delta_vs_nc"] > 0).mean(),
            median_window_kappa_t=inf["kappa_t"].median(),
            share_windows_kappa_t_pos=(inf["kappa_t"].dropna() > 0).mean(),
            yearly_range=complete["macro_f1"].max() - complete["macro_f1"].min(),
            best_year=int(complete.loc[complete["macro_f1"].idxmax(), "year"]) if len(complete) else np.nan,
            worst_year=int(complete.loc[complete["macro_f1"].idxmin(), "year"]) if len(complete) else np.nan,
            **trend_test(yr).to_dict(),
            season_range=se_inf["macro_f1"].max() - se_inf["macro_f1"].min(),
            worst_season=se_inf.loc[se_inf["macro_f1"].idxmin(), "season"] if se_inf["macro_f1"].notna().any() else None,
            offseason_false_alarms_per_30d=off["false_alarms_per_30d"].mean(),
        ))
    out = pd.DataFrame(rows)
    out = (out.merge(event_summary[keys + ["onset_hit_rate", "missed_share", "median_delay_h",
                                           "spurious_episodes_per_year"]], on=keys, how="left")
              .merge(drift_summary, on=keys, how="left")
              .merge(seed_summary[keys + ["macro_f1_std", "mean_band_width"]]
                     .rename(columns={"macro_f1_std": "seed_std_macro_f1", "mean_band_width": "seed_band_width"}),
                     on=keys, how="left"))
    # ranks among learners (1 = best): stable (low IQR), good worst case (high p05), no degradation (high slope)
    learner = out["model"].isin(STAB_LEARNERS)
    grp = out[learner].groupby(["city", "representation"])
    out.loc[learner, "rank_iqr"] = grp["rolling_iqr"].rank()
    out.loc[learner, "rank_p05"] = grp["rolling_p05"].rank(ascending=False)
    out.loc[learner, "rank_trend"] = grp["trend_slope_per_year"].rank(ascending=False)
    out.loc[learner, "stability_rank"] = out.loc[learner, ["rank_iqr", "rank_p05", "rank_trend"]].mean(axis=1)
    out["degrading"] = (out["trend_slope_per_year"] < 0) & (out["trend_p"] < 0.10)
    return out


stability_summary = summarise_stability()
display(stability_summary[keys + ["macro_f1_overall", "rolling_mean", "rolling_iqr", "rolling_p05",
                                  "share_windows_beat_nc", "yearly_range", "trend_slope_per_year", "trend_p",
                                  "worst_season", "onset_hit_rate", "median_delay_h", "error_drifts",
                                  "median_recovery_days", "seed_std_macro_f1", "stability_rank", "degrading"]]
        .sort_values(["city", "representation", "stability_rank"]).round(3))

In [ ]:
# ---- CELL S7: plots ----
REPS = list(REPRESENTATIONS)
GRID_TITLES = [f"{c} – Representation {r}" for c in CITIES for r in REPS]


def _grid_pos(city, rep):
    return CITIES.index(city) + 1, REPS.index(rep) + 1


def _hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{alpha})"


def _band_polygons(band: pd.DataFrame):
    """min/max band -> closed polygons, one per run of non-missing values (None separates them)."""
    ok = band["min"].notna().to_numpy()
    xs, ys = [], []
    for a, b in zip(*_runs(ok)):
        seg = band.iloc[a:b + 1]
        xs += list(seg.index) + list(seg.index[::-1]) + [None]
        ys += list(seg["max"]) + list(seg["min"][::-1]) + [None]
    return xs, ys


def plot_stability_timeline(city: str, rep: str, models=None, filename=None):
    """(1) rolling macro-F1 on informative windows, seed band for ensembles; (2) windowed kappa-t vs persistence;
    (3) event rate with change points of the target (ADWIN) and worsening error drifts."""
    models = models or [m for m in STAB_MODELS if m != MAJ]
    r = rolling[(rolling["city"] == city) & (rolling["representation"] == rep)]
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.45, 0.30, 0.25],
                        subplot_titles=(f"Rolling macro-F1 ({STAB_WINDOW // 24}-day window, informative windows only)",
                                        "Windowed κ-t (0 = as good as repeating the previous label)",
                                        "Event rate in the window · ▲ target change point (ADWIN)"))
    for m in models:
        d = r[r["model"] == m]
        y_f1 = d["macro_f1"].where(d["informative"])
        y_kt = d["kappa_t"].where(d["informative"])
        style = model_style(m)
        if m in STAB_STOCHASTIC:
            sr = seed_rolling[(seed_rolling["city"] == city) & (seed_rolling["representation"] == rep)
                              & (seed_rolling["model"] == m)]
            band = sr.assign(macro_f1=sr["macro_f1"].where(sr["informative"])).groupby("time")["macro_f1"].agg(["min", "max"])
            bx, by = _band_polygons(band)
            fig.add_trace(go.Scatter(x=bx, y=by, mode="lines", line=dict(width=0), fill="toself",
                                     fillcolor=_hex_to_rgba(style["color"], 0.22), name=f"{m}: seed range",
                                     legendgroup=m, showlegend=False, hoverinfo="skip"), row=1, col=1)
        fig.add_trace(go.Scatter(x=d["time"], y=y_f1, mode="lines", name=m, legendgroup=m, line=style,
                                 hovertemplate="%{y:.3f}"), row=1, col=1)
        if m != NC:
            fig.add_trace(go.Scatter(x=d["time"], y=y_kt, mode="lines", name=m, legendgroup=m, showlegend=False,
                                     line=style, hovertemplate="%{y:.2f}"), row=2, col=1)
    fig.add_hline(y=0, line=dict(color="black", width=1, dash="dot"), row=2, col=1)
    d = r[r["model"] == NC]
    fig.add_trace(go.Scatter(x=d["time"], y=d["event_rate"], mode="lines", name="Event rate", fill="tozeroy",
                             line=dict(color="#d62728", width=1), fillcolor="rgba(214,39,40,0.15)",
                             hovertemplate="%{y:.1%}"), row=3, col=1)
    td = target_drifts[target_drifts["city"] == city]
    if len(td):
        fig.add_trace(go.Scatter(x=td["time"], y=[0.02] * len(td), mode="markers", name="Target change point",
                                 marker=dict(symbol="triangle-up", size=8, color="black"),
                                 customdata=np.c_[td["rate_before"], td["rate_after"]],
                                 hovertemplate="rate %{customdata[0]:.1%} → %{customdata[1]:.1%}<extra></extra>"),
                      row=3, col=1)
    fig.update_yaxes(title_text="Macro-F1", row=1, col=1)
    kt = r.loc[r["informative"] & r["model"].isin(models) & (r["model"] != NC), "kappa_t"].dropna()
    kt_low = max(-3.0, float(kt.quantile(0.02))) if len(kt) else -1.0
    fig.update_yaxes(title_text="κ-t (clipped at −3)", range=[kt_low - 0.1, max(1.0, float(kt.max()) + 0.05) if len(kt) else 1],
                     row=2, col=1)
    fig.update_yaxes(title_text="Event rate", tickformat=".0%", rangemode="tozero", row=3, col=1)
    fig.update_layout(title=f"Stability over time – {city}, Representation {rep} "
                            f"(gaps = windows with < {STAB_MIN_EVENT_SHARE:.0%} event hours; band = seed range)",
                      height=900, hovermode="x unified", legend=dict(orientation="h", y=-0.06, x=0))
    show_and_save(fig, filename or f"stability_timeline_{city.lower()}_{rep}")


def plot_yearly_heatmaps(metric="macro_f1", relative=False, filename=None):
    """Model x year. relative=True: difference to No-change in the same year (blue = better than persistence)."""
    col = "delta_vs_nc" if relative else metric
    fig = make_subplots(rows=len(CITIES), cols=len(REPS), subplot_titles=GRID_TITLES,
                        vertical_spacing=0.12, horizontal_spacing=0.12)
    data = yearly
    zmax = np.nanmax(np.abs(data.loc[data["model"].isin(STAB_LEARNERS), col])) if relative else None
    for (city, rep), _ in stab.items():
        d = data[(data["city"] == city) & (data["representation"] == rep)]
        if relative:
            d = d[d["model"].isin(STAB_LEARNERS)]
        piv = d.pivot_table(index="model", columns="year", values=col)
        piv = piv.reindex([m for m in STAB_MODELS if m in piv.index])
        ri, ci = _grid_pos(city, rep)
        fig.add_trace(go.Heatmap(
            z=piv.values, x=[str(c) for c in piv.columns], y=piv.index, text=np.round(piv.values, 2),
            texttemplate="%{text}", textfont=dict(size=10),
            colorscale="RdBu" if relative else "Viridis", zmid=0 if relative else None,
            zmin=-zmax if relative else None, zmax=zmax if relative else None,
            showscale=(ri == 1 and ci == len(REPS)), hovertemplate="%{y}<br>%{x}: %{z:.3f}<extra></extra>"),
            row=ri, col=ci)
        fig.update_yaxes(autorange="reversed", row=ri, col=ci)
    label = f"{metric.replace('_', '-')} minus No-change" if relative else metric.replace("_", "-")
    fig.update_layout(title=f"{label} per year", height=360 * len(CITIES) + 100, margin=dict(l=190))
    show_and_save(fig, filename or f"stability_yearly_{'delta_nc' if relative else metric}")


def plot_seasonal_profile(filename="stability_seasonal"):
    fig = make_subplots(rows=len(CITIES), cols=len(REPS), shared_yaxes=True, subplot_titles=GRID_TITLES,
                        vertical_spacing=0.14, horizontal_spacing=0.04)
    for (city, rep), _ in stab.items():
        ri, ci = _grid_pos(city, rep)
        d = seasonal[(seasonal["city"] == city) & (seasonal["representation"] == rep)]
        for m in STAB_MODELS:
            dm = d[d["model"] == m].set_index("season").reindex(SEASON_ORDER)
            fig.add_trace(go.Bar(x=SEASON_ORDER, y=dm["macro_f1"].where(dm["informative"].fillna(False).astype(bool)), name=m, legendgroup=m,
                                 marker_color=MODEL_COLORS.get(m), showlegend=(ri == 1 and ci == 1),
                                 customdata=np.c_[dm["event_rate"], dm["fp"]],
                                 hovertemplate=f"{m}<br>%{{x}}: %{{y:.3f}}<br>event rate %{{customdata[0]:.1%}}"
                                               "<br>false alarms %{customdata[1]:,}<extra></extra>"),
                          row=ri, col=ci)
    fig.update_yaxes(title_text="Macro-F1", range=[0, 1.02], col=1)
    fig.update_layout(title=f"Macro-F1 per season (all years pooled; seasons with < {STAB_MIN_EVENT_SHARE:.0%} "
                            "event hours left empty) — is quality season-dependent?",
                      barmode="group", height=330 * len(CITIES) + 140, legend=dict(orientation="h", y=-0.12, x=0))
    show_and_save(fig, filename)


def plot_stability_map(filename="stability_map"):
    """x = variability (IQR), y = worst case (p05). Upper-left = stable and reliable."""
    fig = make_subplots(rows=len(CITIES), cols=len(REPS), shared_xaxes=True, shared_yaxes=True,
                        subplot_titles=GRID_TITLES, vertical_spacing=0.12, horizontal_spacing=0.04)
    for (city, rep), _ in stab.items():
        ri, ci = _grid_pos(city, rep)
        d = stability_summary[(stability_summary["city"] == city) & (stability_summary["representation"] == rep)]
        for row in d.itertuples(index=False):
            if row.model == MAJ:
                continue
            family = model_family(row.model) if "model_family" in globals() else "Single learner"
            fig.add_trace(go.Scatter(
                x=[row.rolling_iqr], y=[row.rolling_p05], mode="markers", name=row.model, legendgroup=row.model,
                showlegend=(ri == 1 and ci == 1),
                marker=dict(size=10 + 30 * max(0, row.rolling_mean - 0.5), color=MODEL_COLORS.get(row.model),
                            symbol=FAMILY_SYMBOL.get(family, "circle") if "FAMILY_SYMBOL" in globals() else "circle",
                            line=dict(width=1, color="black")),
                hovertemplate=(f"{row.model}<br>IQR %{{x:.3f}} · p05 %{{y:.3f}}<br>mean {row.rolling_mean:.3f}"
                               f"<br>trend {row.trend_slope_per_year:+.3f}/yr (p={row.trend_p:.2f})"
                               f"<br>beats No-change in {row.share_windows_beat_nc:.0%} of windows<extra></extra>")),
                row=ri, col=ci)
    fig.update_xaxes(title_text="Variability: IQR of rolling macro-F1 (lower = more stable)", row=len(CITIES))
    fig.update_yaxes(title_text="Worst 5 % window (p05)", col=1)
    fig.update_layout(title="Stability map: upper-left = stable and reliable · marker size = mean rolling macro-F1",
                      height=360 * len(CITIES) + 120, legend=dict(orientation="h", y=-0.12, x=0))
    show_and_save(fig, filename)


def plot_event_level(filename="stability_event_level"):
    fig = make_subplots(rows=2, cols=len(CITIES) * len(REPS), shared_yaxes="rows", vertical_spacing=0.14,
                        horizontal_spacing=0.03,
                        subplot_titles=[f"{c} {r}" for c in CITIES for r in REPS] + [""] * len(CITIES) * len(REPS))
    for k, (city, rep) in enumerate([(c, r) for c in CITIES for r in REPS], start=1):
        d = events_yearly[(events_yearly["city"] == city) & (events_yearly["representation"] == rep)]
        for m in STAB_MODELS:
            if m == MAJ:
                continue
            dm = d[d["model"] == m]
            fig.add_trace(go.Scatter(x=dm["year"], y=dm["onset_hit_rate"], mode="lines+markers", name=m, legendgroup=m,
                                     showlegend=(k == 1), line=model_style(m),
                                     hovertemplate=f"{m}<br>%{{x}}: %{{y:.0%}} of onsets caught<extra></extra>"),
                          row=1, col=k)
            fig.add_trace(go.Scatter(x=dm["year"], y=dm["median_delay_h"], mode="lines+markers", name=m, legendgroup=m,
                                     showlegend=False, line=model_style(m),
                                     hovertemplate=f"{m}<br>%{{x}}: median delay %{{y:.0f}} h<extra></extra>"),
                          row=2, col=k)
    fig.update_yaxes(title_text="Onset hit rate", tickformat=".0%", rangemode="tozero", row=1, col=1)
    fig.update_yaxes(title_text="Median detection delay (h)", rangemode="tozero", row=2, col=1)
    fig.update_xaxes(dtick=1, tickformat="d")
    fig.update_layout(title="Event-level stability per year: catching the first hour of an event "
                            "(No-change can never do it) and how late the first alarm comes",
                      height=640, legend=dict(orientation="h", y=-0.12, x=0))
    show_and_save(fig, filename)


def plot_drift_raster(filename="stability_drift_raster"):
    fig = make_subplots(rows=len(CITIES), cols=len(REPS), shared_xaxes=True, subplot_titles=GRID_TITLES,
                        vertical_spacing=0.12, horizontal_spacing=0.1)
    styles = {"worse": dict(symbol="triangle-up", color="#d62728", size=9),
              "better": dict(symbol="triangle-down", color="#2ca02c", size=9)}
    shown = set()
    for (city, rep), _ in stab.items():
        ri, ci = _grid_pos(city, rep)
        d = drifts[(drifts["city"] == city) & (drifts["representation"] == rep) & (drifts["model"] != MAJ)] if len(drifts) else drifts
        for direction, st in styles.items():
            dd = d[(d["kind"] == "error drift") & (d["direction"] == direction)] if len(d) else d
            if len(dd):
                label = f"Error {direction}"
                fig.add_trace(go.Scatter(x=dd["time"], y=dd["model"], mode="markers", name=label, legendgroup=label,
                                         showlegend=label not in shown, marker=st,
                                         customdata=np.c_[dd["error_before"], dd["error_after"], dd["recovery_days"]],
                                         hovertemplate="%{y}<br>error %{customdata[0]:.1%} → %{customdata[1]:.1%}"
                                                       "<br>recovered after %{customdata[2]:.0f} days<extra></extra>"),
                              row=ri, col=ci)
                shown.add(label)
        sw = d[d["kind"] == "ARF tree replaced"] if len(d) else d
        if len(sw):
            fig.add_trace(go.Scatter(x=sw["time"], y=["ARF tree swaps"] * len(sw), mode="markers",
                                     name="ARF tree replaced", legendgroup="arf", showlegend="arf" not in shown,
                                     marker=dict(symbol="line-ns-open", size=10, color="#7f7f7f")), row=ri, col=ci)
            shown.add("arf")
        td = target_drifts[target_drifts["city"] == city]
        if len(td):
            fig.add_trace(go.Scatter(x=td["time"], y=["Target rate (prior)"] * len(td), mode="markers",
                                     name="Target change point", legendgroup="target", showlegend="target" not in shown,
                                     marker=dict(symbol="diamond", size=7, color="black")), row=ri, col=ci)
            shown.add("target")
        for year in pd.DatetimeIndex(stab[(city, rep)]["time"]).year.unique():
            fig.add_vrect(x0=f"{year}-05-01", x1=f"{year}-10-31", fillcolor="rgba(214,39,40,0.06)", line_width=0,
                          layer="below", row=ri, col=ci)
    fig.update_layout(title="When do errors change? ADWIN on each model's 0/1 error stream "
                            "(shaded = hot season May–Oct)", height=420 * len(CITIES) + 100,
                      legend=dict(orientation="h", y=-0.08, x=0), margin=dict(l=190))
    show_and_save(fig, filename)


def plot_seed_spread(filename="stability_seeds"):
    fig = go.Figure()
    for m in STAB_STOCHASTIC:
        d = seed_scores[seed_scores["model"] == m]
        fig.add_trace(go.Box(x=d["city"] + " " + d["representation"], y=d["macro_f1"], name=m, boxpoints="all",
                             marker_color=MODEL_COLORS.get(m), customdata=d["seed"],
                             hovertemplate="seed %{customdata}: %{y:.4f}<extra></extra>"))
    fig.update_layout(title=f"Seed sensitivity of the ensembles ({len(STAB_SEEDS)} seeds, overall macro-F1)",
                      boxmode="group", yaxis_title="Macro-F1", height=440, legend=dict(orientation="h", y=1.1, x=0))
    show_and_save(fig, filename)


def plot_covariate_drift(filename="stability_covariate_drift"):
    fig = make_subplots(rows=1, cols=len(CITIES), subplot_titles=CITIES, horizontal_spacing=0.15)
    for ci, city in enumerate(CITIES, start=1):
        piv = cov_drift[cov_drift["city"] == city].pivot_table(index="feature", columns="year", values="ks_stat")
        fig.add_trace(go.Heatmap(z=piv.values, x=[str(c) for c in piv.columns], y=piv.index, colorscale="Reds",
                                 zmin=0, zmax=max(0.3, np.nanmax(cov_drift["ks_stat"])), text=np.round(piv.values, 2),
                                 texttemplate="%{text}", showscale=(ci == len(CITIES)),
                                 hovertemplate="%{y}<br>%{x}: KS %{z:.3f}<extra></extra>"), row=1, col=ci)
    ref = cov_drift["reference"].iloc[0]
    fig.update_layout(title=f"Input drift: KS distance of each year's hot season to the {ref} hot season "
                            "(0 = same distribution)", height=420, margin=dict(l=170))
    show_and_save(fig, filename)


for city in CITIES:
    for rep in REPS:
        plot_stability_timeline(city, rep)
plot_stability_map()
plot_yearly_heatmaps("macro_f1")
plot_yearly_heatmaps(relative=True)
plot_seasonal_profile()
plot_event_level()
plot_drift_raster()
plot_seed_spread()
plot_covariate_drift()

In [ ]:
# ---- CELL S8: evidence-based findings (a draft to verify against the plots, then rewrite in your own words) ----
def _fmt(x, f="{:.3f}"):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f.format(x)


def stability_findings() -> str:
    s = stability_summary
    lines = ["### Stability findings (generated from the tables above)\n"]
    for city in CITIES:
        for rep in REPS:
            d = s[(s["city"] == city) & (s["representation"] == rep)]
            L = d[d["model"].isin(STAB_LEARNERS)].sort_values("stability_rank")
            nc = d[d["model"] == NC].iloc[0]
            best, worst = L.iloc[0], L.iloc[-1]
            lines.append(f"**{city}, Representation {rep}**")
            lines.append(f"- Most stable learner: **{best.model}** (IQR {_fmt(best.rolling_iqr)}, worst 5 % window "
                         f"{_fmt(best.rolling_p05)}, yearly range {_fmt(best.yearly_range)}). Least stable: "
                         f"{worst.model} (IQR {_fmt(worst.rolling_iqr)}, p05 {_fmt(worst.rolling_p05)}).")
            lines.append(f"- No-change: IQR {_fmt(nc.rolling_iqr)}, p05 {_fmt(nc.rolling_p05)}. Learners beat it in "
                         f"{_fmt(L.share_windows_beat_nc.min(), '{:.0%}')}–{_fmt(L.share_windows_beat_nc.max(), '{:.0%}')} "
                         f"of informative windows; median windowed κ-t of the best learner: "
                         f"{_fmt(best.median_window_kappa_t, '{:+.2f}')}.")
            deg = L[L["degrading"]]
            if len(deg):
                lines.append("- **Degrading over the years** (OLS slope < 0, p < 0.10): " + "; ".join(
                    f"{r.model} {r.trend_slope_per_year:+.3f}/yr (p={r.trend_p:.2f}, {r.best_year}→{r.worst_year})"
                    for r in deg.itertuples()) + ".")
            else:
                lines.append("- No learner shows a significant downward trend across years (p < 0.10).")
            lines.append(f"- Worst season for most learners: {L['worst_season'].mode().iat[0] if L['worst_season'].notna().any() else 'n/a'}; "
                         f"off-season false alarms range {_fmt(L.offseason_false_alarms_per_30d.min(), '{:.1f}')}–"
                         f"{_fmt(L.offseason_false_alarms_per_30d.max(), '{:.1f}')} h per 30 days.")
            on = L.sort_values("onset_hit_rate", ascending=False)
            lines.append("- Event onsets (No-change catches 0 % by construction): " + "; ".join(
                f"{r.model} {_fmt(r.onset_hit_rate, '{:.0%}')} of first hours, median delay "
                f"{_fmt(r.median_delay_h, '{:.0f}')} h, {_fmt(r.missed_share, '{:.0%}')} episodes missed, "
                f"{_fmt(r.spurious_episodes_per_year, '{:.1f}')} spurious alarm episodes/yr"
                for r in on.itertuples()) + ". Read onset hits together with spurious alarms: firing often buys onsets.")
            ed = drifts[(drifts["city"] == city) & (drifts["representation"] == rep)
                        & (drifts["kind"] == "error drift") & drifts["model"].isin(STAB_LEARNERS)] if len(drifts) else drifts
            if len(ed):
                month = pd.DatetimeIndex(stab[(city, rep)]["time"]).month
                expected = float(np.isin(month, TRANSITION_MONTHS).mean())        # share of hours in those months
                k = int(ed["in_transition_month"].sum())
                p_bin = sps.binomtest(k, len(ed), expected, alternative="greater").pvalue
                worse = ed[ed["direction"] == "worse"]
                rec = worse["recovery_days"]
                lines.append(
                    f"- Error drifts (ADWIN, learners): {len(ed)} ({len(worse)} worsening); {k / len(ed):.0%} in "
                    f"season-transition months vs {expected:.0%} expected by chance (binomial p = {p_bin:.3f}) → "
                    + ("drift is mainly **seasonal**." if p_bin < 0.05 else "no evidence that drift concentrates at season changes.")
                    + (f" Worsening drifts: {int(rec.isna().sum())} not recovered within {RECOVERY_MAX_DAYS} days; the rest "
                       f"recovered after a median {_fmt(rec.median(), '{:.0f}')} days"
                       + (f" (= the {RECOVERY_WINDOW // 24}-day resolution: immediate)." if rec.median() <= RECOVERY_WINDOW / 24 + 0.5 else ".")
                       if len(worse) else ""))
            for r in L[L["model"] == "Adaptive Random Forest"].itertuples():
                n_sw = int(np.nan_to_num(getattr(r, "arf_tree_replacements", 0) or 0))
                lines.append(f"- Adaptive Random Forest replaced {n_sw} trees during the stream (its own ADWIN detectors)"
                             + (" — evidence of active adaptation." if n_sw else " — it never needed to adapt."))
            seeds = L[L["seed_std_macro_f1"].notna()]
            if len(seeds):
                lines.append(f"- Seed sensitivity: std of overall macro-F1 across {len(STAB_SEEDS)} seeds ≤ "
                             f"{_fmt(seeds.seed_std_macro_f1.max(), '{:.4f}')} — "
                             + ("negligible next to the differences between models." if seeds.seed_std_macro_f1.max() < 0.01
                                else "not negligible: rankings between close models are not robust."))
            lines.append("")
    pr = prior_drift.pivot_table(index="city", columns="year", values="event_rate")
    lines.append("**Data drift**")
    for city in CITIES:
        row = pr.loc[city].dropna()
        shift = row.max() - row.min()
        lines.append(f"- {city}: yearly event rate {row.min():.1%} ({row.idxmin()}) – {row.max():.1%} ({row.idxmax()}); "
                     + (f"a {shift * 100:.1f} pp swing with τ_hot frozen = **prior-probability shift**."
                        if shift >= 0.03 else "stable event rate (< 3 pp swing)."))
    top_cov = cov_drift.sort_values("ks_stat", ascending=False).head(3)
    lines.append("- Largest input shifts (hot season, KS): " + "; ".join(
        f"{r.city} {r.feature} {r.year} ({r.ks_stat:.2f})" for r in top_cov.itertuples()) + ".")
    ws = window_sensitivity[window_sensitivity["model"].isin(STAB_LEARNERS)]
    lines.append(f"- Window length: the IQR ranking of learners agrees across {', '.join(str(w // 24) for w in STAB_WINDOW_GRID)}"
                 f"-day windows with mean Kendall τ = {np.nanmean(rank_agreement.values[np.triu_indices(len(rank_agreement), 1)]):.2f}.")
    return "\n".join(lines)


findings_md = stability_findings()
display(Markdown(findings_md))

In [ ]:
# ---- CELL S9: exports for the report ----
exports = {
    "stability_summary.csv": stability_summary, "stability_rolling.csv": rolling,
    "stability_yearly.csv": yearly, "stability_seasonal.csv": seasonal, "stability_year_season.csv": year_season,
    "stability_events.csv": events, "stability_event_summary.csv": event_summary,
    "stability_events_yearly.csv": events_yearly, "stability_alarm_episodes.csv": alarms,
    "stability_drifts.csv": drifts, "stability_target_drifts.csv": target_drifts,
    "stability_drift_summary.csv": drift_summary, "stability_covariate_drift.csv": cov_drift,
    "stability_seed_scores.csv": seed_scores, "stability_seed_summary.csv": seed_summary,
    "stability_window_sensitivity.csv": window_sensitivity, "stability_coefficients.csv": coef_stability,
}
for name, df in exports.items():
    df.to_csv(STAB_DIR / name, index=False)
(STAB_DIR / "stability_findings.md").write_text(findings_md, encoding="utf-8")
pd.DataFrame({"parameter": ["window_h", "stride_h", "min_event_share", "min_transitions", "seeds", "adwin_delta",
                            "recovery_window_h", "recovery_max_days", "hot_season_months", "window_grid_h", "tuned"],
              "value": [STAB_WINDOW, STAB_STRIDE, STAB_MIN_EVENT_SHARE, STAB_MIN_TRANSITIONS, STAB_SEEDS,
                        ADWIN_DELTA, RECOVERY_WINDOW, RECOVERY_MAX_DAYS, HOT_SEASON_MONTHS, STAB_WINDOW_GRID,
                        STAB_USE_TUNED]}).to_csv(STAB_DIR / "stability_configuration.csv", index=False)
print(f"Saved {len(exports) + 2} files to {STAB_DIR}")

### Stability analysis — written interpretation

*Write this from the plots and from the generated findings above; every claim should point to a figure or table.*

1. **Is any learner stable enough to deploy?** Compare IQR and p05 to No-change (stability map, summary table).
2. **Do models degrade over the years?** Trend slopes and the yearly heatmap; connect any degradation to the
   prior shift (event rate per year) and to input drift (KS heatmap).
3. **Is the instability seasonal or structural?** Seasonal bars, share of ADWIN drifts in transition months,
   recovery times.
4. **Where do the models add value over persistence?** Windowed κ-t and onset hit rate — persistence can never
   catch the first hour of an event.
5. **Do representation B and the ensembles make the models more stable, not only better on average?**
   Compare IQR/p05/slope between A and B and between Hoeffding Tree and the ensembles.
6. **How robust are these conclusions?** Seed spread and window-length sensitivity.
7. **Consequence for the recommendation** (Online Logistic Regression B): does its coefficient settling time and
   stability ranking support it?